
# Transformation Exploration Notebook

This notebook allows you to explore the transformation logic interactively.
You can load the extracted datasets, inspect raw data, and test transformation steps.


In [ ]:
import pandas as pd
import numpy as np

# just to show it works

In [ ]:
# Load raw datasets (ensure extract.py has been run)
retail = pd.read_csv('RetailSales_2026-01.csv')
orders = pd.read_csv('OrderBank_2026-01.csv')

retail.head(), orders.head()


In [ ]:
retail.info()

In [ ]:
orders.info()

In [ ]:
# Basic cleaning
retail_clean = retail.copy()
orders_clean = orders.copy()

for col in ["country", "dealer_id", "model", "powertrain"]:
    retail_clean[col] = retail_clean[col].astype(str).str.strip()
    orders_clean[col] = orders_clean[col].astype(str).str.strip()

# Convert dates
temp = pd.to_datetime(retail_clean['report_month'])
retail_clean['report_month'] = temp.dt.to_period('M').astype(str)

orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'])
orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)


In [ ]:

retail_clean['units_sold'] = pd.to_numeric(retail_clean['units_sold'], errors='coerce')
retail_clean['revenue_local'] = pd.to_numeric(retail_clean['revenue_local'], errors='coerce')
orders_clean['final_price_local'] = pd.to_numeric(orders_clean['final_price_local'], errors='coerce')



In [ ]:
retail_clean.info(), orders_clean.info()

In [ ]:

order_summary = (
    orders_clean
    .groupby(["order_month", "country", "dealer_id", "model", "powertrain"])
    .agg(order_count=("order_id", "count"),
         avg_configured_price=("final_price_local", "mean"))
    .reset_index()
)

order_summary.head()


In [ ]:

gold = pd.merge(
    retail_clean,
    order_summary,
    left_on=["report_month", "country", "dealer_id", "model", "powertrain"],
    right_on=["order_month", "country", "dealer_id", "model", "powertrain"],
    how="left"
)

gold = gold.drop(columns=["order_month"], errors="ignore")

gold['avg_selling_price'] = gold['revenue_local'] / gold['units_sold']

gold.head()


### Sales Performance Overview (Gold Layer)
Business Insight: Identify which models, dealers, countries, and powertrains generate the most revenue and units sold.

In [ ]:
gold.groupby(["country", "model"]) \
    .agg(total_units=("units_sold", "sum"),
         total_revenue=("revenue_local", "sum")) \
    .sort_values("total_revenue", ascending=False)

### Demand vs. Sales Gap (Order Bank vs. Retail Sales)
Business Insight: Compare order_count with units_sold to identify:

-high demand but low fulfilment (potential supply constraints)

-low demand but high sales (stock clearance, old inventory)

In [ ]:
gold["demand_gap"] = gold["order_count"] - gold["units_sold"]

gold[["country", "dealer_id", "model", "demand_gap"]] \
    .sort_values("demand_gap", ascending=False).head(10)

### Price Sensitivity & Customer Behaviour
Business Insight: Using avg_configured_price (from Order Bank) vs. avg_selling_price (from Sales):

-identify premium‑leaning markets (high configuration spend)

-detect price‑sensitive regions (low configured price)

-assess how optional features influence average selling price

In [ ]:
gold[["country", "model", "avg_configured_price", "avg_selling_price"]] \
    .groupby(["country", "model"]).mean() \
    .sort_values("avg_configured_price", ascending=False)